# v11 A (GPU 0): count-matched lesion and silencing on FINISHED models (no training)
In every layer n_max = size of the smaller group (self or task). Compared: n_max random self neurons vs n_max random task neurons (and half of that, plus look-alike task neurons at half).
- **lesion** = weights re-drawn as at initialisation; **silence** = neuron held at its average.
- per layer (one layer at a time) and all 8 layers at once; 2 draws; loss rise on 200 val pairs per language; chrF (100 FLORES devtest sentences) for all 8 layers.
- Models: distant phases 10, 11, 12, 40, 41, 42 · Hindi-only 10, 30, 45, 60 · mixed 10, 30, 45, 60 · related 10-15.
Results: `analysis/v11/static/` · progress: `tail -f logs/v11_A_static.log`

In [1]:
import os, sys, glob, json, time, subprocess
os.environ["HF_HUB_DISABLE_XET"] = "1"
GPU = "0"; LOG = "v11_A_static"
REPO = "/workspace/shreshth"; os.chdir(REPO); sys.path.insert(0, REPO)
from IPython.display import Image, display
from huggingface_hub import HfApi, hf_hub_download
FILES, HF_REPO = [], None
for attempt in range(5):
    try:
        api = HfApi(); HF_REPO = api.whoami()["name"] + "/cml-lang-self-checkpoints"; FILES = api.list_repo_files(HF_REPO); break
    except Exception as e:
        print("HF retry", attempt, type(e).__name__); time.sleep(10)
if not FILES:
    print("could not list the HF repo: only checkpoints already on this machine will be used")

def have(run, ph):
    return bool(glob.glob(f"runs/{run}/ckpt/p{ph:03d}_*.pt"))

def get(run, phases):
    """download what is missing; returns the phases that are on this machine afterwards"""
    ok = []
    for ph in phases:
        if not have(run, ph):
            name = [f for f in FILES if f.startswith(f"{run}/ckpt/p{ph:03d}_")]
            if name:
                try:
                    hf_hub_download(HF_REPO, name[0], local_dir="runs")
                except Exception as e:
                    print("download failed:", run, ph, type(e).__name__, e)
        if have(run, ph):
            ok.append(ph)
        else:
            print("not available (not here, not on HF):", run, ph)
    return ok

def run(langset, *cmd):
    """one call of core_v11.py; output shown here AND appended to logs/<LOG>.log (watch with: tail -f logs/<LOG>.log)"""
    env = {**os.environ, "CML_LANGSET": langset}; os.makedirs("logs", exist_ok=True)
    p = subprocess.Popen([sys.executable, "-u", "core_v11.py", "--gpu", GPU, *cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    with open(f"logs/{LOG}.log", "a") as log:
        for line in p.stdout:
            print(line, end=""); log.write(line); log.flush()
    p.wait(); print("exit code", p.returncode)

def summary(part, out=None):
    cmd = [sys.executable, "core_v11.py", "--summarize", "--part", part] + (["--out", out] if out else [])
    r = subprocess.run(cmd, capture_output=True, text=True); print(r.stdout[-12000:]); print(r.stderr[-3000:])

DIST, HIN, MIX, REL = "v4b_continual_s0", "v4b_only_hi_s0", "v6b_mixed_s0", "v8_indic_continual_s0"

/venv/main/lib/python3.12/site-packages/huggingface_hub/constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


In [2]:
if os.path.exists("analysis/v11/static/static.csv"):
    os.remove("analysis/v11/static/static.csv")                    # this part has no resume: start clean
PLAN = [("distant", "distant", DIST, [10, 11, 12, 40, 41, 42], None), ("distant", "hindi", HIN, [10, 30, 45, 60], "hi"),
        ("distant", "mixed", MIX, [10, 30, 45, 60], None), ("indic", "related", REL, [10, 11, 12, 13, 14, 15], None)]
for langset, fam, r, phs, ev in PLAN:
    ok = get(r, phs)
    if ok:
        cmd = ["--part", "static", "--family", fam, "--model", f"{fam}={r}:" + ",".join(str(p) for p in ok)]
        run(langset, *(cmd + (["--eval_langs", ev] if ev else [])))

distant phase 10 (hi): intact loss hi 1.107, zh 6.441, ar 6.015 | chrF hi 54.1, zh 2.7, ar 9.8
  count-matched n per layer (n_max = smaller group): enc0:1247 enc1:1133 enc2:1273 enc3:1008 dec0:1229 dec1:1039 dec2:1108 dec3:897


  all 8 layers, loss rise (average of languages and draws):
cond     look@0.5  self@0.5  self@1.0  task@0.5  task@1.0
mode                                                     
lesion      0.270     0.539     2.157     0.241     1.585
silence     0.108     0.332     1.721     0.099     1.216
  restored model check: loss differs from intact by 0.00000 (should be ~0) | 111s


distant phase 11 (zh): intact loss hi 4.349, zh 2.062, ar 6.105 | chrF hi 25.6, zh 26.3, ar 13.2
  count-matched n per layer (n_max = smaller group): enc0:1253 enc1:1137 enc2:1275 enc3:998 dec0:1201 dec1:1132 dec2:1084 dec3:1135


  all 8 layers, loss rise (average of languages and draws):
cond     look@0.5  self@0.5  self@1.0  task@0.5  task@1.0
mode                                                     
lesion      0.221     0.521     2.939     0.195     0.552
silence     0.146     0.407     2.962     0.134     0.375
  restored model check: loss differs from intact by 0.00000 (should be ~0) | 108s


distant phase 12 (ar): intact loss hi 5.924, zh 5.565, ar 1.731 | chrF hi 7.2, zh 3.7, ar 53.4
  count-matched n per layer (n_max = smaller group): enc0:1265 enc1:1197 enc2:1242 enc3:925 dec0:1150 dec1:1161 dec2:1082 dec3:504


  all 8 layers, loss rise (average of languages and draws):
cond     look@0.5  self@0.5  self@1.0  task@0.5  task@1.0
mode                                                     
lesion      0.224     0.422     1.720     0.163     0.894
silence     0.081     0.259     1.403     0.061     0.586
  restored model check: loss differs from intact by 0.00000 (should be ~0) | 106s


... (34 outputs left out)

  all 8 layers, loss rise (average of languages and draws):
cond     look@0.5  self@0.5  self@1.0  task@0.5  task@1.0
mode                                                     
lesion      0.275     0.515     2.078     0.208     0.686
silence     0.177     0.424     1.804     0.151     0.462
  restored model check: loss differs from intact by 0.00000 (should be ~0) | 101s


related phase 15 (pa): intact loss hi 3.069, gu 2.947, pa 1.873 | chrF hi 40.5, gu 38.2, pa 43.6
  count-matched n per layer (n_max = smaller group): enc0:1176 enc1:1152 enc2:1203 enc3:1258 dec0:851 dec1:1249 dec2:1249 dec3:674


  all 8 layers, loss rise (average of languages and draws):
cond     look@0.5  self@0.5  self@1.0  task@0.5  task@1.0
mode                                                     
lesion      0.237     0.490     2.028     0.206     0.648
silence     0.156     0.417     1.894     0.138     0.434
  restored model check: loss differs from intact by 0.00000 (should be ~0) | 95s
done


exit code 0


In [3]:
summary("static")

=== static (no training): loss rise on the model's languages; one comparison = one checkpoint x one scope x one draw (average of the languages) ===

--- self vs task (count-matched)
 family        scope    mode dose   self   task  ratio self_worse  sign_p
distant all 8 layers  lesion  0.5 0.5224 0.2309 2.2622      12/12  0.0002
distant all 8 layers  lesion  1.0 2.2444 1.0849 2.0688      12/12  0.0002
distant all 8 layers silence  0.5 0.3487 0.1222 2.8531      12/12  0.0002
distant all 8 layers silence  1.0 1.7803 0.7361 2.4187      12/12  0.0002
distant    one layer  lesion  0.5 0.0409 0.0241 1.6984      75/96  0.0000
distant    one layer  lesion  1.0 0.1170 0.0851 1.3753      80/96  0.0000
distant    one layer silence  0.5 0.0275 0.0118 2.3395      76/96  0.0000
distant    one layer silence  1.0 0.0830 0.0560 1.4841      71/96  0.0000
  hindi all 8 layers  lesion  0.5 0.6260 0.3516 1.7804        8/8  0.0039
  hindi all 8 layers  lesion  1.0 2.9884 1.2666 2.3594        8/8  0.0039
  hi